# Ask Compass 01 - Setup

Run this notebook once for each frozen Classroom Compass snapshot. It builds the normalized local registry, reconstructs the report-aligned baseline, prepares local Store C from original teacher prose, generates Store A/B/C documents, creates versioned OpenAI vector stores, uploads Store A and Store B, and writes one setup manifest for the chat notebook.

**Boundary:** the local registry remains the analytical source of truth. Store A adds semantic recall. Store B holds methodology/reference context. Store C is project evidence only and cannot create or modify a Compass finding.

**Store C upload is intentionally OFF by default.** The notebook still builds the complete local Store C and creates the empty vector store. Set `UPLOAD_STORE_C = True` only after the retention/data-governance decision is cleared.

For raw essays, either put the eight SQL Runner CSV exports (or `esssays.zip`) in the project root, put the validated Store C bundle in the project root, or set `RAW_ESSAY_SOURCE` explicitly below.

In [1]:
from pathlib import Path
import os
import getpass
import json
import pandas as pd

import ask_compass_utils as ac

# ---------------------------------------------------------------------
# EDIT THIS ONE PATH
# ---------------------------------------------------------------------
ROOT = Path('/Users/matt.fritz/Desktop/Research Insights/Essay Prototype')

# Optional explicit source. Leave None to auto-discover esssays.zip,
# ask_compass_store_c_local_bundle.zip, or an extracted Store C lookup.
RAW_ESSAY_SOURCE = None

# Vector-store setup
CREATE_VECTOR_STORES = True
UPLOAD_STORE_A = True
UPLOAD_STORE_B = True
UPLOAD_STORE_C = True
FORCE_REBUILD_STORES = False
VECTOR_STORE_EXPIRES_DAYS = 90

# Matches the current Compass local proxy convention. Revisit before deployment.
VERIFY_SSL = False

print('Root:', ROOT)

Root: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype


In [2]:
# API key is used only for vector-store creation/upload. It is never written to disk.
if CREATE_VECTOR_STORES and not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')

print('API key available:', bool(os.getenv('OPENAI_API_KEY')))

API key available: True


In [3]:
manifest = ac.run_setup(
    root=ROOT,
    raw_essay_source=RAW_ESSAY_SOURCE,
    create_vector_stores=CREATE_VECTOR_STORES,
    upload_store_a=UPLOAD_STORE_A,
    upload_store_b=UPLOAD_STORE_B,
    upload_store_c=UPLOAD_STORE_C,
    force_rebuild_stores=FORCE_REBUILD_STORES,
    vector_store_expires_days=VECTOR_STORE_EXPIRES_DAYS,
    verify_ssl=VERIFY_SSL,
)

ac.setup_summary(manifest)

Ask Compass snapshot: 20260526_b5958b0bb9
Report: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/ask_compass_prototype_handoff/00_report_data.json
Frozen baseline: {'cutoff': '2026-05-26', 'project_count': 2242059, 'target_report_project_count': 2175476, 'exact_count_match': False, 'warning': 'Cutoff count does not exactly match the frozen report. Treat baseline deltas as weak application signals until extract/version drift is reconciled.'}
Local Store C: {'ready': True, 'expected_relationships': 52350, 'expected_unique_projects': 39286, 'returned_unique_projects': 39286, 'missing_unique_projects': 0, 'unexpected_unique_projects': 0, 'min_essays_per_insight': 50, 'max_essays_per_insight': 50, 'prose_like_sample_share': 1.0, 'source': '/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/essays.zip'}
Registry: {'registry_version': '0.1.0', 'record_count': 1047, 'report_insight_count': 1047, 'full_bridge_rows': 2055601, 'full_bridge_unique_projects': 586738, '

,store,vector_store_id,local_documents,upload_requested,upload_complete
0,A,vs_6ab1957da8048191a7b36ea26200eefb,1047,True,True
1,B,vs_6ab19db63e2081918650b33d4db8bf28,1,True,True
2,C,vs_6ab19dbe96b08191a73a86349584276b,1047,True,True


## Setup checks

The checks below are deliberately small. They confirm that the manifest points to a complete registry, that Store A can semantically retrieve an approved insight, and that Store B can retrieve the operating reference. They do not evaluate final answer quality; that happens in Notebook 02.

In [6]:
print(json.dumps({
    'snapshot_id': manifest['snapshot_id'],
    'baseline': manifest['baseline'],
    'registry_record_count': manifest['registry']['record_count'],
    'store_c_local': manifest['store_c_local'],
}, indent=2))

if CREATE_VECTOR_STORES:
    client = ac.get_openai_client(verify_ssl=VERIFY_SSL)
    stores = manifest.get('vector_stores', {})

    if (stores.get('A') or {}).get('upload_complete'):
        print('\nSTORE A SMOKE TEST')
        hits = ac.search_vector_store(
            client,
            vector_store_id=stores['A']['id'],
            query='math manipulatives and visible math thinking',
            max_num_results=5,
            rewrite_query=True,
        )
        for h in hits[:5]:
            print(round(float(h.get('score') or 0), 4), (h.get('attributes') or {}).get('insight_id'))

    if (stores.get('B') or {}).get('upload_complete'):
        print('\nSTORE B SMOKE TEST')
        hits = ac.search_vector_store(
            client,
            vector_store_id=stores['B']['id'],
            query='How should Ask Compass interpret attributes and evidence gaps?',
            max_num_results=3,
            rewrite_query=True,
        )
        for h in hits[:3]:
            print(round(float(h.get('score') or 0), 4), (h.get('text') or '')[:300].replace('\n', ' '))

    if store_c_id and (stores.get('C') or {}).get('upload_complete'):
        client = ac.get_openai_client(verify_ssl=VERIFY_SSL)
        insight_id = 'stem__strategic_injected_tag__missing__ki_002__math_concepts_are_taught_through_physical_models_first'
        hits = ac.search_vector_store(
            client,
            vector_store_id=store_c_id,
            query='hands-on math manipulatives that help students understand abstract math concepts',
            max_num_results=5,
            rewrite_query=True,
            attribute_filter={'type': 'eq', 'key': 'insight_id', 'value': insight_id},
        )
        print('STORE C SMOKE TEST')
        for hit in hits:
            print(round(float(hit.get('score') or 0), 4), hit.get('filename'))
            print((hit.get('text') or '')[:600].replace('\n', ' '))
            print('---')


{
  "snapshot_id": "20260526_b5958b0bb9",
  "baseline": {
    "cutoff": "2026-05-26",
    "project_count": 2242059,
    "target_report_project_count": 2175476,
    "exact_count_match": false,
    "warning": "Cutoff count does not exactly match the frozen report. Treat baseline deltas as weak application signals until extract/version drift is reconciled."
  },
  "registry_record_count": 1047,
  "store_c_local": {
    "ready": true,
    "bridge_path": "/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/ask_compass_agent/snapshots/20260526_b5958b0bb9/store_c_local/store_c_top50_bridge.csv.gz",
    "essay_lookup_path": "/Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/ask_compass_agent/snapshots/20260526_b5958b0bb9/store_c_local/store_c_essay_lookup.csv.gz",
    "expected_relationships": 52350,
    "expected_unique_projects": 39286,
    "returned_unique_projects": 39286,
    "missing_unique_projects": 0,
    "unexpected_unique_projects": 0,
    "min_essa

## Output contract

The only artifact Notebook 02 needs is:

`OUTPUTS/ask_compass_agent/setup_manifest.json`

Rerunning this notebook with the same frozen report reuses completed vector stores unless `FORCE_REBUILD_STORES=True`. A new report snapshot gets a new versioned snapshot directory and new store names.

## Store C smoke test

Runs only when Store C was uploaded. The search is filtered to one approved insight, so essays remain downstream of insight selection.


STORE C SMOKE TEST
0.987 projects_6138a11d60d6cdcf.md
## PROJECT_ID: 8759770 COMPASS_PROJECT_RANK: 32 math manipulatives enhance learning in the classroom by providing tangible, hands-on tools that help students understand abstract mathematical concepts. these tools—like fraction tiles, buildable blocks, and immersive games—allow students to visually and physically explore and manipulate mathematical ideas, making them more concrete. this hands-on approach helps students grasp concepts such as addition, subtraction, multiplication, and fractions more deeply by engaging multiple senses and facilitating active learning. manipulatives also support d
---
0.978 projects_6138a11d60d6cdcf.md
 1. providing handation will directly impact the education of our students by:g but also develop a deeper conceptual understanding of place value and arithmetic operations. these tools will enable them to explore mathematical concepts with confidence and enthusiasm, laying a solid foundation for future ma